# Figures 5–7: Pt/Co FTH, interaction modes, and spectral imaging

At normal incidence, a 300 nm Au mask has an object hole of radius 180 nm and a reference hole of radius 30 nm, separated by 600 nm. Beneath the object hole lie SiN(20)/Pt(2)/Co(10)/Pt(2), all in nm. The reference hole traverses the entire stack. Separating the holes by more than $3R_{object}+R_{reference}=570$ nm separates the ideal reference cross correlation from the central autocorrelation. The grid is 192×192 at 10 nm pitch.

The code resolves Au in 10 nm slices, SiN in 5 nm slices, and Co in 2 nm slices. It runs scalar, Jones and coherent-carrier Stokes propagation for matched CR and CL illumination. The far field is an orthonormal FFT on a uniform transverse momentum grid. The reconstruction is the shifted inverse FFT of **intensity CR−CL**; the displayed sideband magnitude is reference-blurred, and discards its complex sign/phase. It is not claimed to be a quantitative inversion of magnetization.

The modes use the same specimen and beam. Scalar is an eigenmode approximation; agreement is expected here because the tabulated linear channel is zero and the normal-incidence transverse response has circular eigenmodes. Stokes with pure input shares a Jones carrier, so agreement is a consistency check, not independent physical validation.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/scattering_calculator").is_dir())
PAPER = ROOT / "paper/scattering_calculator"
sys.path.insert(0, str(PAPER))
import experiments as ex
OUT = PAPER / "results"
OUT.mkdir(exist_ok=True)

def show(name):
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.imshow(plt.imread(OUT / (name + ".png")))
    ax.axis("off")
    plt.show()


## Editable sample and geometry

Change these values and Run All. The settings below override the reference numbers in the introductory prose. The FTH calculation uses a uniform momentum grid; physical detector pitch and distance are configured in the skyrmion notebooks.

In [ ]:
# EDIT HERE. Film/aperture dimensions and sampling are in nm.
sample_cfg = ex.FTHConfig(
    energy_eV=778.0,
    n=192, dx_nm=10.0,
    layers_nm=(('Au',300.), ('SiN',20.), ('Pt',2.), ('Co',10.), ('Pt',2.)),
    max_slice_nm=5.0,
    object_radius_nm=180.0,
    reference_radius_nm=30.0,
    reference_xy_nm=(600.0, 0.0),
    beam_sigma_nm=650.0,
    domain_period_nm=110.0,
    propagate=True,
    energies_eV=(772.,775.,778.,780.,783.,790.,795.,800.),
)
OUT = PAPER / 'results' / 'notebook_fth'
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
ex.provenance(OUT, {'notebook': 'paper_02'})
metrics = ex.fth_figures(OUT, config=sample_cfg)
print(metrics)
show('fig05_fth')
show('fig06_modes')

## Co L-edge spectral stack on a common q grid

The energy list is 772, 775, 778, 780, 783, 790, 795, 800 eV. These are a sparse illustrative selection within the loader's Co circular-contrast window, not a spectroscopy-resolution claim. We hold real-space pixel size and illuminated amplitude fixed, so the transverse momentum grid stays fixed. This corresponds to regridded data, not the same pixels on a stationary flat detector at every energy. For an actual fixed detector, recompute $q(E)$ and interpolate before comparing reconstructions.

The plot reports the RMS complex amplitude in the reference sideband, the tabulated circular coefficients, and an energy-position slice through the reconstructed images. Absorption and interference also affect the sideband: its spectrum is not simply the absorption coefficient. Raw complex reconstructions and CR−CL holograms are saved in `hyperspectral.npz`.


In [ ]:
show('fig07_hyperspectral')
spectral = np.load(OUT / 'hyperspectral.npz')
print('Energy axis (eV):', spectral['energies_eV'])
print('Reconstruction cube:', spectral['reconstruction'].shape)